## LOGISTIC REGRESSION WITH SMOTE & SMOTEN SAMPLING TECHNIQUE AND TFIDF

In [3]:
#imporing dependeincies
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from imblearn.over_sampling import SMOTEN,SMOTE,ADASYN,RandomOverSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
import mlflow
import mlflow.sklearn

C:\Users\siawc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# loading the data
data = pd.read_csv(r'C:\Users\siawc\OneDrive\Desktop\Felix\mental health\eda& preprocessing\health_folder\mental_heath_unbanlanced_cleaned.csv')
data.head(5)

,status,cleaned_text
0,Anxiety,oh gosh
1,Anxiety,trouble sleeping confused mind restless heart ...
2,Anxiety,wrong back dear forward doubt stay restless re...
3,Anxiety,ive shifted focus something else im still worried
4,Anxiety,im restless restless month boy mean


In [5]:
#dropping the null values
data.dropna(inplace=True)

In [6]:
encoder = LabelEncoder()
data['status'] = encoder.fit_transform(data['status'])

data['status'].value_counts()

status
2    18317
1    14505
3    11209
0     5503
Name: count, dtype: int64

In [7]:
# splitting the data into train,test and evaluation data
X = data.drop('status',axis=1)
y = data['status']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

X_val,X_test,y_val,y_test = train_test_split(X_test,y_test, test_size=0.5,random_state=42)

In [8]:
# converting word to vec
def convert_word_to_vec(X_train, X_test):
    try:
        # Ensure all input features are strings before TF-IDF
        X_train = X_train.astype(str)
        X_test = X_test.astype(str)

        try:
            vectorizer = TfidfVectorizer(
              
            )
            X_train_vec = vectorizer.fit_transform(X_train.values.ravel())
        except ValueError as e:
            if 'empty vocabulary' in str(e).lower():
                vectorizer = TfidfVectorizer(
                    max_features=500,
                    ngram_range=(1,2),
                    max_df=0.7
                )
                X_train_vec = vectorizer.fit_transform(X_train.values.ravel())
            else:
                raise

        X_test_vec = vectorizer.transform(X_test.values.ravel())
        return X_train_vec, X_test_vec, vectorizer
    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None, None


In [9]:
# applying the sampling techniques
def sample_techniques(X_train, y_train, techniques):
 
    try:
        if techniques == 'smote':
            sampler = RandomOverSampler(sampling_strategy='auto', random_state=42)
            X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)
            print("RandomOverSampler success")
        else:
            adasyn = ADASYN(
                sampling_strategy='auto',  # oversample all minority classes
                random_state=42,
                n_neighbors=3 
            )
            X_train_res, y_train_res = adasyn.fit_resample(X_train, y_train)
        return X_train_res, y_train_res
    except Exception as e:
        print(f'An error occurred: {e}')
        import traceback
        traceback.print_exc()
        return X_train, y_train  # return original data if sampling fails

In [10]:
mlflow.set_experiment('Mental Health Status')
mlflow.set_tracking_uri('http://127.0.0.1:5000')

2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/13 23:24:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/13 23:24:45 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/13 23:24:45 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [11]:
print("Testing RandomOverSampler on full data")
try:
    sampler = RandomOverSampler(sampling_strategy='auto', random_state=42)
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    print("Success, new shape:", X_res.shape, y_res.shape)
    print("New y value_counts:")
    print(pd.Series(y_res).value_counts())
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Testing RandomOverSampler on full data
Success, new shape: (51144, 1) (51144,)
New y value_counts:
status
2    12786
3    12786
1    12786
0    12786
Name: count, dtype: int64


In [12]:
def train_the_model(X_train,y_train,X_test,y_test,X_val,y_val):
    """
    Convert the raw text data to vectors, train a logistic regression classifier and evaluate 
    on held-out splits. Metrics are logged to MLflow for easy comparison.

    Parameters
    ----------
    X_train, X_test, X_val : pandas Series/DataFrame of text
    y_train, y_test, y_val : array-like labels
    """
    try:
        with mlflow.start_run() as run:
            # converting the training data and testing data to vectors
            X_train_vec, X_test_vec, vectorizer = convert_word_to_vec(X_train, X_test)
            if X_train_vec is None or X_test_vec is None or vectorizer is None:
                raise ValueError('Vectorizer fit failed; check input text and column schema')
            X_val_vec = vectorizer.transform(X_val.values.ravel())
            
            # performing oversampling techniques
            X_train_res1,y_train_res1 = sample_techniques(X_train_vec,y_train,techniques='smote')
            
            

            # training
            solver = 'lbfgs'
            max_iter = 500
            multi_class = 'multinomial'
        
            
            mlflow.log_param('Solver',solver)
            mlflow.log_param('Multi_Class',multi_class)
            
            mlflow.log_param('Max_iter',max_iter)
            
            # smote training
            model_1 = LogisticRegression(
                solver=solver,
                max_iter=max_iter,
                C=1

            )
            model_1.fit(X_train_res1,y_train_res1)

            test_pred1 = model_1.predict(X_test_vec)
            eval_pred1 = model_1.predict(X_val_vec)

            test_accuracy = accuracy_score(y_test,test_pred1)
            eval_accuracy = accuracy_score(y_val,eval_pred1)
            
            test_report1 = classification_report(y_test,test_pred1,output_dict=True)
            eval_report1 = classification_report(y_val,eval_pred1, output_dict=True)

            for label,metrics in test_report1.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'test_{label}_{metric}',value)

            for label,metrics in eval_report1.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'eval_{label}_{metric}',value)
            
            mlflow.sklearn.log_model('Logistic Regression',model_1)
            print(f'Testing Accuracy for SMOTE: {test_accuracy}')
            print(f'Evaluation Accuracy for SMOTE: {eval_accuracy}')

    except Exception as e:
        print(f'An error occured {e}')

In [ ]:
train_the_model(X_train,y_train,X_test,y_test,X_val,y_val)